### 1.1.5. Unordered Sets

An **unordered set** in AMPL is a finite collection of distinct elements that carries no position
information. The elements may be numbers, strings, or tuples (ordered pairs, triples, etc.), but
the set itself imposes no sequence on them. Unordered sets are the default: every set is unordered
unless its declaration includes the keyword `ordered` or `circular`.

The simplest unordered set declaration contains only the keyword `set` and a name. No ordering
attribute is present, so the set is unordered by definition. The examples below show the three
structural forms — declaration only, symbolic definition, and membership constraint — all of which
produce unordered sets.

In [ ]:
set PROD;               # Form 1: declaration only — members come from data or amplpy
set SHIFTS = {"Day", "Evening", "Night"};   # Form 2: symbolic definition — fixed at model-generation time
set CHEAP within PROD;  # Form 3: membership constraint — every data-supplied member must be in PROD

# Form 4: 2-D unordered set — dimension inferred from 'within' expression
set ORIG;  set DEST;
set LINKS within ORIG cross DEST;

# Form 5: unordered set with explicit dimension (no 'within' available yet)
set ROUTES dimen 3;   # members are triples (a, b, c)

# None of the above carry 'ordered', 'ordered by', 'circular', or 'circular by'.
# All are unordered. Membership, cardinality, and set operators are valid.

Set names follow the same rules as other AMPL model components (parameters, variables, constraints). A name may consist of letters, digits, and underscores (_), and must not be one of AMPL's [reserved](<Reserved names.ipynb>) names.

**Default status.** Unless a set declaration includes `ordered` or `circular`, the set is
unordered. This is true regardless of whether the members are strings or numbers, whether the
set is simple (1-D) or compound (multi-dimensional), and whether it is static (data-supplied) or
symbolic (defined by an expression).

**Enumeration order is implementation-defined.** When AMPL enumerates the members of an
unordered set — for example, in a `display` command or when generating indexed constraints —
the order in which members appear is not specified by the language. It may sometimes coincide with data input order, but this behavior is not specified and must never be relied upon. A model that is correct for any permutation of an unordered set is a model that is
correct.

**Compound sets are always unordered.** Sets of dimension two or higher — pairs, triples, or
tuples — cannot be declared `ordered` or `circular`(this is a syntax restriction in AMPL). This restriction applies regardless of how
the compound set was constructed (explicit listing, `cross`, `within`, or `setof`).

In [ ]:
# Unordered 2-D set with explicit dimension:
set LINKS dimen 2;
set LINKS within ORIG cross DEST;
?????
set LINKS dimen 2 within ORIG cross DEST;   # dimension inferred as 2

**Arithmetic ranges are manifestly ordered.** The expression `a .. b` or `a .. b by k` in a
model is a *manifestly ordered* set — AMPL treats it as ordered (and the position-dependent
functions apply to it: such as first, last, prev, next, ord) even without an explicit `ordered` declaration. However, a set *defined*
by such an expression (`set YEARS = 2010 .. 2040 by 5;`) is still officially unordered unless
the declaration adds `ordered`. The `diff` operation preserves the ordering of its left operand
when that operand is ordered; `union`, `inter`, and `symdiff` of ordered sets yield unordered
results.

**Functions valid on unordered sets:**

| Function / operator | Syntax | Returns |
|---|---|---|
| Cardinality | `card(S)` | Number of members in `S` (integer) |
| Membership test | `e in S` | Boolean: 1 if `e` is a member of `S`, else 0 |
| Non-membership test | `e not in S` | Boolean: 1 if `e` is not a member of `S`, else 0 |
| Subset test | `A within B` | Boolean: 1 if A ⊆ B, else 0 |
| Union | `A union B` | Set of members in `A` or `B` |
| Intersection | `A inter B` | Set of members in both `A` and `B` |
| Difference | `A diff B` | Set of members in `A` but not `B` |
| Symmetric difference | `A symdiff B` | Set of members in exactly one of `A`, `B` |
| Cartesian product | `A cross B` | Set of all tuples `(a,b)` with `a in A`, `b in B` |
| Set comprehension | `setof {i in S: cond} expr` | Set of values of `expr` for members of `S` satisfying `cond` |

In [ ]:
# ---- UNORDERED version (cannot reference prev or first) ----

set PROD;
set WEEKS;          # unordered — no 'ordered' keyword

param make  {PROD, WEEKS} >= 0;
param inv0  {PROD}        >= 0;
var Make    {PROD, WEEKS} >= 0;
var Inv     {PROD, WEEKS} >= 0;
var Sell    {PROD, WEEKS} >= 0;

# The following constraint references prev(t), which requires WEEKS to be ordered.
# With an unordered WEEKS this is a MODEL ERROR at generation time:
#
# subject to Balance {p in PROD, t in WEEKS: ord(t) > 1}:
#     Make[p,t] + Inv[p,prev(t)] = Sell[p,t] + Inv[p,t];
#
# AMPL will report: "prev applied to unordered set WEEKS"


# ---- ORDERED version (prev and first are valid) ----

set WEEKS2 ordered;  # ordering keyword present

param inv02 {PROD}         >= 0;
var Make2  {PROD, WEEKS2}  >= 0;
var Inv2   {PROD, WEEKS2}  >= 0;
var Sell2  {PROD, WEEKS2}  >= 0;

subject to Balance0 {p in PROD}:
    Make2[p, first(WEEKS2)] + inv02[p]
    = Sell2[p, first(WEEKS2)] + Inv2[p, first(WEEKS2)];

subject to Balance {p in PROD, t in WEEKS2: ord(t) > 1}:
    Make2[p,t] + Inv2[p, prev(t)] = Sell2[p,t] + Inv2[p,t];

#### Example 1 — Valid Operations on Unordered Sets

This example demonstrates the full set of operations that are valid on unordered sets:
membership test, cardinality, and the five binary set operators. All of these work regardless
of whether the set is ordered.

In [ ]:
# Unordered sets of products — members supplied from data
set MINREQ;           # nutrients with a minimum daily requirement
set MAXREQ;           # nutrients with a maximum daily requirement

# All set operators produce unordered results when applied to unordered sets
set NUTR    = MINREQ union  MAXREQ;   # all nutrients (duplicates excluded)
set BOTH    = MINREQ inter  MAXREQ;   # nutrients with both constraints
set MINONLY = MINREQ diff   MAXREQ;   # nutrients with only a minimum
set MAXONLY = MAXREQ diff   MINREQ;   # nutrients with only a maximum
set EITHER  = MINREQ symdiff MAXREQ;  # nutrients in exactly one constraint set

# Cardinality: valid on unordered sets
param n_nutr = card(NUTR);

# Membership test: valid on unordered sets
param has_cal = if 'CAL' in NUTR then 1 else 0;

#### Example 2 — Symbolic, Numeric, and Tuple Sets (All Unordered by Default)

Unordered sets can hold strings, numbers, or tuples. The type of element does not affect
ordering status. This example shows all three element types and confirms that the same
operations — and only those operations — are valid on each.

In [ ]:
# String (symbolic) set — unordered
set PROD;                                   # e.g., {"bands", "coils", "plate"}

# Numeric set — unordered (distinct from an arithmetic progression, which is ordered)
set YEARS;                                  # e.g., {2020, 2025, 2030} from data

# Arithmetic progression — manifestly ordered by construction
set RANGE = 2010 .. 2040 by 5;              # The expression is ordered, but RANGE itself is NOT an ordered set unless declared with 'ordered'

# Tuple (2-D) set — always unordered; ordered/circular not permitted
set ORIG;  
set DEST;
set LINKS within ORIG cross DEST;           # sparse subset of (orig, dest) pairs

# Valid on all three:
display card(PROD);                         # number of products
display card(YEARS);                        # number of years
display card(LINKS);                        # number of links

# Membership test (valid on all):
# 'bands' in PROD, 2025 in YEARS, ('GARY','DET') in LINKS

# NOT valid on any of them (would be a model error):
# first(PROD), last(YEARS), next('bands', PROD), ord(2025, YEARS)
# first(LINKS)  -- also invalid because LINKS is 2-D, and even ordered
#               -- is not permitted on sets of dimension >= 2

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
Declare a set <code>ordered</code> only when the model explicitly requires position-dependent
operations — <code>first</code>, <code>last</code>, <code>next</code>, <code>prev</code>,
<code>ord</code>, or <code>ord0</code>. Every other set should remain unordered. Unnecessary
ordering declarations do not improve model correctness; they create an obligation to handle
boundary members (the first member has no predecessor, the last has no successor) and force
data to be supplied in a specific sequence.</div>

<div class="alert alert-block alert-success"> <b>Best practice: </b><br>
When writing constraints or sums indexed over an unordered set, ensure that the constraint
body is correct for <em>any</em> permutation of the set members. If the correctness of a
constraint depends on the enumeration order, the set must be declared ordered — relying on
implementation-defined iteration order is a latent bug that may survive testing but fail on
a different AMPL build or platform.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Calling <code>first(S)</code>, <code>last(S)</code>, <code>next(e, S)</code>,
<code>prev(e, S)</code>, <code>ord(e, S)</code>, or <code>ord0(e, S)</code> on an
unordered set is a model error. AMPL will report an error at model-generation time
(not at solve time), such as: <em>"prev applied to unordered set S"</em>. The fix is
either to add <code>ordered</code> to the set declaration (if position semantics are
genuinely required) or to remove the position-dependent call (if the model logic does not
actually need it).</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
The result of <code>union</code>, <code>inter</code>, and <code>symdiff</code> on two ordered
sets is an <strong>unordered</strong> set. Only <code>diff</code> preserves the ordering of its
left operand when that operand is ordered (AMPL Reference Manual, A.6.2). Consequently, if you
compute a derived set using <code>union</code> or <code>inter</code> of ordered sets and then
try to call <code>first</code> or <code>prev</code> on the result, AMPL will report an error.
To get an ordered derived set from ordered operands, use the <code>ordered by</code> phrase
in its declaration to inherit the ordering from a containing ordered set.</div>

<div class="alert alert-block alert-warning"> <b>Pitfall: </b><br>
Sets of dimension two or higher cannot be declared <code>ordered</code> or <code>circular</code>.
The restriction applies universally: a set of pairs, triples, or higher-dimensional tuples is
always unordered, regardless of how it was constructed. Attempting to add <code>ordered</code>
to a 2-D set declaration is a syntax error.</div>

#### Edge Cases

**Edge case 1: Applying ordered-set functions to an unordered set is a model error, not a runtime error.**

AMPL checks the ordering attribute of a set at model-generation time, when it processes the
declaration. Calling `first(S)`, `prev(e, S)`, or any other position-dependent function on an
unordered set produces an error immediately at model generation — not when a solve is attempted.
This means the error is caught early, before any data is loaded.

In [ ]:
# This model fragment would produce an error at generation time:

set PERIODS;            # unordered — no 'ordered' keyword
var x {PERIODS} >= 0;

# ERROR: prev applied to unordered set PERIODS
# subject to Link {t in PERIODS: ord(t) > 1}:
#     x[t] >= x[prev(t)];

# Fix: declare the set ordered
set PERIODS2 ordered;
var y {PERIODS2} >= 0;

subject to Link {t in PERIODS2: ord(t) > 1}:
    y[t] >= y[prev(t)];   # valid: PERIODS2 is ordered

**Edge case 2: `union` and `inter` of ordered sets produce unordered results.**

When two ordered sets are combined using `union`, `inter`, or `symdiff`, the result is
unordered — AMPL does not propagate the ordering attribute through these operators.
Only `diff` preserves the ordering of its left operand. This is documented in the AMPL
Reference Manual (Section A.6.2).

The consequence is that a derived set computed as `A union B` cannot be used with `first`,
`prev`, or `ord` even if `A` and `B` are both ordered.

In [ ]:
set ALL_WEEKS ordered;          # ordered: data-supply order determines sequence
param T1 integer > 0;
param T2 integer > T1;

# Ordered subset via 'ordered by' — inherits ordering from ALL_WEEKS
set EARLY ordered by ALL_WEEKS = {t in ALL_WEEKS: ord(t) <= T1};
set LATE  ordered by ALL_WEEKS = {t in ALL_WEEKS: ord(t) >  T1};

# diff preserves ordering of left operand:
set NOT_LAST ordered by ALL_WEEKS = ALL_WEEKS diff {last(ALL_WEEKS)};
# first(NOT_LAST) and prev(t, NOT_LAST) are valid.

# union of two ordered sets yields an UNORDERED result:
set BOTH_HALVES = EARLY union LATE;   # unordered — ordering NOT preserved
# first(BOTH_HALVES) would be a model error.

# To get an ordered union of EARLY and LATE:
# use 'ordered by ALL_WEEKS' on a set that is defined as a subset of ALL_WEEKS.
set ALL_EXPLICIT ordered by ALL_WEEKS = {t in ALL_WEEKS: ord(t) <= T2};
# first(ALL_EXPLICIT) is valid.

**Edge case 3: Arithmetic progressions are manifestly ordered but a set declaration still determines the set's ordering attribute.**

The expression `2010 .. 2040 by 5` is a *manifestly ordered* set expression — AMPL will retain
its ordering when computing derived sets. However, if a named set is *defined* by this expression
without an `ordered` phrase in its declaration, the named set is officially unordered:
position-dependent functions cannot be applied to it by name. Position-dependent functions operate on declared sets, not on raw set expressions.

This is a subtle distinction: the expression itself carries ordering information that AMPL uses
internally (for example, in `diff` operations whose left operand is a range), but the named set
does not inherit ordered status without an explicit declaration.

In [ ]:
param T_start integer;
param T_end   integer > T_start;

# Without 'ordered': YEARS is defined by an ordered expression but is itself unordered
set YEARS = T_start .. T_end by 5;
# first(YEARS) is a model error.

# With 'ordered': YEARS2 is ordered; position functions are valid
set YEARS2 ordered = T_start .. T_end by 5;
# first(YEARS2) is valid and returns T_start.
# prev(t, YEARS2) is valid for any t in YEARS2 except the first member.

**Edge case 4: An empty unordered set is valid everywhere an unordered set is valid.**

An unordered set may be empty at any point during a model run. Indexing a constraint over an
empty set generates zero constraint instances — this is not an error. All set operators produce
well-defined results when one or both operands are empty, consistent with the standard mathematical
definitions. Cardinality of the empty set is 0. The empty set is a subset of every set
(`{} within S` is always true).

In [ ]:
set MINREQ2 = {'protein', 'fat', 'sodium', 'cal'};
set MAXREQ2 = {};                    # empty for this instance

set NUTR2    = MINREQ2 union  MAXREQ2;   # = MINREQ2; no error
set BOTH2    = MINREQ2 inter  MAXREQ2;   # = {}; no error
set EITHER2  = MINREQ2 symdiff MAXREQ2;  # = MINREQ2; no error

display card(NUTR2);    # 4
display card(BOTH2);    # 0

# Subset test: the empty set is a subset of every set
display MAXREQ2 within NUTR2;   # 1 (true)
display MAXREQ2 within BOTH2;   # 1 (true)

# Constraint indexed over empty set generates zero instances — not an error
param n_max {MAXREQ2} >= 0;
param amt_   {NUTR2, {'dummy_food'}} >= 0 default 0;
var  Buy_ >= 0;
subject to DietMax {i in MAXREQ2}:    # zero constraints generated; model is valid
    sum {j in {'dummy_food'}} amt_[i,j] * Buy_ <= n_max[i];